# SMAX checkpoint evaluation

This notebook evaluates saved SMAX checkpoints with deterministic rollouts and reports paper-table win rates at the training-seed level.


In [9]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "main.py").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root from the current notebook working directory.")


ROOT = resolve_repo_root()
ARTIFACT_ROOT = ROOT
FIGURES_DIR = ARTIFACT_ROOT / "figures"

CHECKPOINTS_DIR = ARTIFACT_ROOT / "checkpoints_for_eval_SMAX"

CHECKPOINTS_DIR = "/home/chamjin/research/pci_marl/checkpoints_for_eval_SMAX"

for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

command_buffer_flag = "--xla_gpu_enable_command_buffer="
existing_xla_flags = os.environ.get("XLA_FLAGS", "")
if command_buffer_flag not in existing_xla_flags:
    os.environ["XLA_FLAGS"] = f"{existing_xla_flags} {command_buffer_flag}".strip()
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import tab2_smax_checkpoint_eval_utils as scu
scu = importlib.reload(scu)

bootstrap_project = scu.bootstrap_project
discover_runs = scu.discover_runs
evaluate_map = scu.evaluate_map
evaluate_map_constraint = scu.evaluate_map_constraint
format_cross_map_latex = scu.format_cross_map_latex
ordered_map_names = scu.ordered_map_names
ordered_model_labels = scu.ordered_model_labels
save_summary_tables = scu.save_summary_tables
save_full_evaluation_bundle = scu.save_full_evaluation_bundle

DEVICE = "cuda"
CHECKPOINT_CHOICE = "final"  # or "best"
DUPLICATE_POLICY = "latest"  # curated checkpoints_for_eval_SMAX notebook policy; utility default remains "error"
EVAL_SEEDS = list(range(10001, 10065))
EVAL_EPISODES = 1
EVAL_SEED_BATCH_SIZE = 16
MAP_RESULTS = {}

bootstrap_project(DEVICE)
runs_df = discover_runs(CHECKPOINTS_DIR, project_filter=None)
if runs_df.empty:
    raise RuntimeError("No runs were discovered under checkpoints_for_eval_SMAX")

In [10]:
# Select Maps and Models

TARGET_MAPS = ordered_map_names(runs_df["map_name"].dropna().unique().tolist()) # auto
TARGET_MAPS = ['6h_vs_8z'] # we can select map here!

TARGET_MODELS = ordered_model_labels(runs_df["model_label"].dropna().unique().tolist())

print(f"Repo root: {ROOT}")
print(f"Loaded {len(runs_df)} evaluation runs from checkpoints_for_eval_SMAX")
print(f"Maps: {TARGET_MAPS}")
print(f"Models: {TARGET_MODELS}")
display(runs_df.groupby(["map_name", "constraint_label", "model_label"]).size().unstack(fill_value=0))
print("Cached evaluations are reused automatically when checkpoint path, mtime, eval seed, and episode count match.")

Repo root: /home/chamjin/research/pci_marl
Loaded 180 evaluation runs from checkpoints_for_eval_SMAX
Maps: ['6h_vs_8z']
Models: ['ippo', 'pci-ippo', 'mappo', 'pci-mappo']


model_label                       ippo  mappo  pci-ippo  pci-mappo
map_name        constraint_label                                  
2s3z            tar_w=3              3      3         3          3
                tar_w=4              3      3         3          3
                vanilla              3      3         3          3
3m              tar_w=2              3      3         3          3
                vanilla              3      3         3          3
3s5z            tar_w=6              3      3         3          3
                vanilla              3      3         3          3
6h_vs_8z        tar_w=6              3      3         3          3
                vanilla              3      3         3          3
smacv2_10_units tar_w=6              3      3         3          3
                tar_w=8              3      3         3          3
                vanilla              3      3         3          3
smacv2_5_units  tar_w=3              3      3         3          3
                tar_w=4              3      3         3          3
                vanilla              3      3         3          3

Cached evaluations are reused automatically when checkpoint path, mtime, eval seed, and episode count match.


In [11]:
SMOKE_EVAL_SEEDS = EVAL_SEEDS[:2]

smoke_row = runs_df.sort_values(["map_name", "constraint_key", "model_label", "seed", "run_dir"]).iloc[0]
SMOKE_MAP = smoke_row["map_name"]
SMOKE_CONSTRAINT = smoke_row["constraint_key"]
SMOKE_MODEL = smoke_row["model_label"]
SMOKE_SEED = int(smoke_row["seed"])

smoke_runs = runs_df[
    (runs_df["map_name"] == SMOKE_MAP)
    & (runs_df["constraint_key"] == SMOKE_CONSTRAINT)
    & (runs_df["model_label"] == SMOKE_MODEL)
    & (runs_df["seed"] == SMOKE_SEED)
].copy()

display(smoke_runs[["map_name", "constraint_key", "model_label", "seed", "wandb_run_name", "run_dir"]])

smoke_result = evaluate_map_constraint(
    smoke_runs,
    SMOKE_MAP,
    SMOKE_CONSTRAINT,
    checkpoint_choice=CHECKPOINT_CHOICE,
    device=DEVICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=SMOKE_EVAL_SEEDS,
    eval_seed_batch_size=min(EVAL_SEED_BATCH_SIZE, len(SMOKE_EVAL_SEEDS)),
    duplicate_policy=DUPLICATE_POLICY,
)
display(smoke_result["run_results"])
display(smoke_result["seed_summary"])
display(smoke_result["model_summary"])


,map_name,constraint_key,model_label,seed,wandb_run_name,run_dir
0,2s3z,tar_w=3,ippo,1,smax_2s3z_tar_w=3_Fixmask,/home/chamjin/research/pci_marl/checkpoints_fo...


/tmp/ipykernel_3697642/3041507688.py:18: UserWarning: std_success_rate is defined over training seeds. Filled 0.0 for rows with n_seeds=1.
map_name constraint_key model_label  n_seeds
    2s3z        tar_w=3        ippo        1
  smoke_result = evaluate_map_constraint(


,algorithm_type,checkpoint_choice,checkpoint_path,constraint_key,constraint_label,eval_max_reward,eval_mean_length,eval_mean_reward,eval_min_reward,eval_seed,...,map_name,metadata_path,model_key,model_label,num_episodes,project,run_dir,seed,train_seed,wandb_run_name
0,jax_ippo,final,/home/chamjin/research/pci_marl/checkpoints_fo...,tar_w=3,tar_w=3,2.0,56.0,2.0,2.0,10001,...,2s3z,/home/chamjin/research/pci_marl/checkpoints_fo...,ippo,ippo,1,pci-marl-smax-jax,/home/chamjin/research/pci_marl/checkpoints_fo...,1,1,smax_2s3z_tar_w=3_Fixmask
1,jax_ippo,final,/home/chamjin/research/pci_marl/checkpoints_fo...,tar_w=3,tar_w=3,2.0,67.0,2.0,2.0,10002,...,2s3z,/home/chamjin/research/pci_marl/checkpoints_fo...,ippo,ippo,1,pci-marl-smax-jax,/home/chamjin/research/pci_marl/checkpoints_fo...,1,1,smax_2s3z_tar_w=3_Fixmask


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,1,1.0,2.0,61.5,1,2


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,1.0,0.0,2.0,NaN,61.5,NaN,1,1,2


## Full evaluation

This cell evaluates each target map with a shared eval-seed set and preserves the aggregation shape:

- `run_results`: each train seed x eval seed result
- `seed_summary`: each train seed averaged over eval seeds
- `model_summary`: mean $\pm$ std over training seeds


In [12]:
MAP_RESULTS = {}
for map_name in TARGET_MAPS:
    result = evaluate_map(
        runs_df,
        map_name,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
        eval_seeds=EVAL_SEEDS,
        eval_seed_batch_size=EVAL_SEED_BATCH_SIZE,
        model_order=TARGET_MODELS,
        duplicate_policy=DUPLICATE_POLICY,
    )
    if result["model_summary"].empty:
        print(f"Skipping {map_name}: no discovered runs")
        continue
    MAP_RESULTS[map_name] = result
    print(f"\n=== {map_name} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    print(result["latex"])


[SMAXWrapper] '6h_vs_8z': 6 allies vs 8 enemies
[SMAXBatchWrapper] '6h_vs_8z': 6v8, n_envs=16
[SMAXBatchWrapper] obs_dim=179, action_dim=13
[SMAXBatchWrapper] 6v8 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=179, effective_obs_dim=185
[SMAXWrapper] '6h_vs_8z': 6 allies vs 8 enemies
[SMAXBatchWrapper] '6h_vs_8z': 6v8, n_envs=16
[SMAXBatchWrapper] obs_dim=179, action_dim=13
[SMAXBatchWrapper] 6v8 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=179, effective_obs_dim=185
[SMAXWrapper] '6h_vs_8z': 6 allies vs 8 enemies
[SMAXBatchWrapper] '6h_vs_8z': 6v8, n_envs=16
[SMAXBatchWrapper] allowed_targets (full source-level damage mask, window=6): {0: [0, 1, 2, 3, 4, 5], 1: [1, 2, 3, 4, 5, 6], 2: [3, 4, 5, 6, 7, 0], 3: [4, 5, 6, 7, 0, 1], 4: [5, 6, 7, 0, 1, 2], 5: [7, 0, 1, 2, 3, 4]}
[SMAXBatchWrapper] obs_dim=179, action_dim=13
[SMAXBatchWrapper] 6v8 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=179, effective_obs_dim=185
[SMAXWrapper] '6h_vs_8z': 6 allies vs 8 enemies
[SMAXBatc

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,1,0.796875,1.761797,41.875000,1,64
1,6h_vs_8z,vanilla,vanilla,ippo,2,0.984375,1.983984,33.218750,1,64
2,6h_vs_8z,vanilla,vanilla,ippo,42,0.203125,0.995781,35.765625,1,64
3,6h_vs_8z,vanilla,vanilla,pci-ippo,1,1.000000,2.000000,34.531250,1,64
4,6h_vs_8z,vanilla,vanilla,pci-ippo,2,0.968750,1.960312,52.656250,1,64
5,6h_vs_8z,vanilla,vanilla,pci-ippo,42,0.984375,1.981484,55.250000,1,64
6,6h_vs_8z,vanilla,vanilla,mappo,1,0.906250,1.896797,39.093750,1,64
7,6h_vs_8z,vanilla,vanilla,mappo,2,0.984375,1.983984,33.109375,1,64
8,6h_vs_8z,vanilla,vanilla,mappo,42,0.906250,1.898828,41.546875,1,64
9,6h_vs_8z,vanilla,vanilla,pci-mappo,1,1.000000,2.000000,31.875000,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,0.661458,0.407849,1.580521,5.184420e-01,36.953125,4.448627,3,3,64
1,6h_vs_8z,vanilla,vanilla,pci-ippo,0.984375,0.015625,1.980599,1.985854e-02,47.479167,11.287971,3,3,64
2,6h_vs_8z,vanilla,vanilla,mappo,0.932292,0.045105,1.926536,4.976182e-02,37.916667,4.340161,3,3,64
3,6h_vs_8z,vanilla,vanilla,pci-mappo,1.000000,0.000000,2.000000,1.493992e-08,32.677083,1.010846,3,3,64
4,6h_vs_8z,tar_w=6,tar_w=6,ippo,0.453125,0.291899,1.358854,3.491679e-01,42.838542,5.867106,3,3,64
5,6h_vs_8z,tar_w=6,tar_w=6,pci-ippo,0.869792,0.212140,1.849635,2.467656e-01,46.494792,10.365505,3,3,64
6,6h_vs_8z,tar_w=6,tar_w=6,mappo,0.838542,0.252591,1.817943,2.848890e-01,38.182292,1.067124,3,3,64
7,6h_vs_8z,tar_w=6,tar_w=6,pci-mappo,0.807292,0.168528,1.790911,1.835380e-01,43.348958,6.283238,3,3,64


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 6h_vs_8z using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_6h_vs_8z_final}
\begin{tabular}{lcccc}
\toprule
Constraint & ippo & pci-ippo & mappo & pci-mappo \\
\midrule
vanilla & 66.1 \scriptsize $\pm$ 40.8 & 98.4 \scriptsize $\pm$ 1.6 & 93.2 \scriptsize $\pm$ 4.5 & 100.0 \scriptsize $\pm$ 0.0 \\
tar_w=6 & 45.3 \scriptsize $\pm$ 29.2 & 87.0 \scriptsize $\pm$ 21.2 & 83.9 \scriptsize $\pm$ 25.3 & 80.7 \scriptsize $\pm$ 16.9 \\
\bottomrule
\end{tabular}
\end{table}


In [13]:
for map_name, result in MAP_RESULTS.items():
    run_df = result["run_results"].copy()
    seed_summary = result["seed_summary"].copy()
    model_summary = result["model_summary"].copy()

    eval_seed_sets = (
        run_df.groupby(["constraint_key", "model_label", "train_seed"])["eval_seed"]
        .agg(lambda values: tuple(sorted(set(int(v) for v in values))))
        .reset_index(name="eval_seed_set")
    )
    method_seed_counts = eval_seed_sets.groupby(["constraint_key", "model_label"])["eval_seed_set"].nunique()
    assert (method_seed_counts == 1).all(), f"Inconsistent eval_seed sets detected for {map_name}:\n{method_seed_counts}"

    assert not (run_df["train_seed"] == run_df["eval_seed"]).any(), f"train_seed and eval_seed collided for {map_name}"

    expected_seed_rows = (
        run_df.groupby(["constraint_key", "model_label"])["train_seed"]
        .nunique()
        .reset_index(name="expected_rows")
    )
    actual_seed_rows = (
        seed_summary.groupby(["constraint_key", "model_label"])
        .size()
        .reset_index(name="actual_rows")
    )
    row_check = expected_seed_rows.merge(actual_seed_rows, on=["constraint_key", "model_label"], how="left")
    assert (row_check["expected_rows"] == row_check["actual_rows"]).all(), f"seed_summary row mismatch for {map_name}:\n{row_check}"

    expected_std = (
        seed_summary.groupby(["constraint_key", "model_label"])["eval_success_rate"]
        .std()
        .reset_index(name="expected_std_success_rate")
    )
    std_check = model_summary.merge(expected_std, on=["constraint_key", "model_label"], how="left")
    observed = std_check["std_success_rate"].fillna(0.0).to_numpy()
    expected = std_check["expected_std_success_rate"].fillna(0.0).to_numpy()
    assert ((observed - expected).__abs__() < 1.0e-12).all(), f"std_success_rate mismatch for {map_name}:\n{std_check}"

    if DUPLICATE_POLICY != "all":
        assert (seed_summary["n_runs"] == 1).all(), f"duplicate runs leaked into seed_summary for {map_name}:\n{seed_summary[['constraint_key', 'model_label', 'seed', 'n_runs']]}"

print("Protocol assertions passed.")


Protocol assertions passed.


In [14]:
INSPECT_MAP = TARGET_MAPS[0]
if INSPECT_MAP not in MAP_RESULTS:
    raise RuntimeError(f"Run the full evaluation cell first. Missing {INSPECT_MAP}.")

display(MAP_RESULTS[INSPECT_MAP]["seed_summary"])
display(MAP_RESULTS[INSPECT_MAP]["model_summary"])
print(MAP_RESULTS[INSPECT_MAP]["latex"])


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,1,0.796875,1.761797,41.875000,1,64
1,6h_vs_8z,vanilla,vanilla,ippo,2,0.984375,1.983984,33.218750,1,64
2,6h_vs_8z,vanilla,vanilla,ippo,42,0.203125,0.995781,35.765625,1,64
3,6h_vs_8z,vanilla,vanilla,pci-ippo,1,1.000000,2.000000,34.531250,1,64
4,6h_vs_8z,vanilla,vanilla,pci-ippo,2,0.968750,1.960312,52.656250,1,64
5,6h_vs_8z,vanilla,vanilla,pci-ippo,42,0.984375,1.981484,55.250000,1,64
6,6h_vs_8z,vanilla,vanilla,mappo,1,0.906250,1.896797,39.093750,1,64
7,6h_vs_8z,vanilla,vanilla,mappo,2,0.984375,1.983984,33.109375,1,64
8,6h_vs_8z,vanilla,vanilla,mappo,42,0.906250,1.898828,41.546875,1,64
9,6h_vs_8z,vanilla,vanilla,pci-mappo,1,1.000000,2.000000,31.875000,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,0.661458,0.407849,1.580521,5.184420e-01,36.953125,4.448627,3,3,64
1,6h_vs_8z,vanilla,vanilla,pci-ippo,0.984375,0.015625,1.980599,1.985854e-02,47.479167,11.287971,3,3,64
2,6h_vs_8z,vanilla,vanilla,mappo,0.932292,0.045105,1.926536,4.976182e-02,37.916667,4.340161,3,3,64
3,6h_vs_8z,vanilla,vanilla,pci-mappo,1.000000,0.000000,2.000000,1.493992e-08,32.677083,1.010846,3,3,64
4,6h_vs_8z,tar_w=6,tar_w=6,ippo,0.453125,0.291899,1.358854,3.491679e-01,42.838542,5.867106,3,3,64
5,6h_vs_8z,tar_w=6,tar_w=6,pci-ippo,0.869792,0.212140,1.849635,2.467656e-01,46.494792,10.365505,3,3,64
6,6h_vs_8z,tar_w=6,tar_w=6,mappo,0.838542,0.252591,1.817943,2.848890e-01,38.182292,1.067124,3,3,64
7,6h_vs_8z,tar_w=6,tar_w=6,pci-mappo,0.807292,0.168528,1.790911,1.835380e-01,43.348958,6.283238,3,3,64


\begin{table}[t]
\centering
\caption{Deterministic SMAX win rate on 6h_vs_8z using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_6h_vs_8z_final}
\begin{tabular}{lcccc}
\toprule
Constraint & ippo & pci-ippo & mappo & pci-mappo \\
\midrule
vanilla & 66.1 \scriptsize $\pm$ 40.8 & 98.4 \scriptsize $\pm$ 1.6 & 93.2 \scriptsize $\pm$ 4.5 & 100.0 \scriptsize $\pm$ 0.0 \\
tar_w=6 & 45.3 \scriptsize $\pm$ 29.2 & 87.0 \scriptsize $\pm$ 21.2 & 83.9 \scriptsize $\pm$ 25.3 & 80.7 \scriptsize $\pm$ 16.9 \\
\bottomrule
\end{tabular}
\end{table}


In [15]:
if not MAP_RESULTS:
    raise RuntimeError("Run the full evaluation cell first.")

combined_model_summary = pd.concat(
    [MAP_RESULTS[map_name]["model_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_model_summary)

latex_code = format_cross_map_latex(
    combined_model_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(latex_code)

paths = save_summary_tables(
    combined_model_summary,
    FIGURES_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(f"Saved CSV: {paths['csv']}")
print(f"Saved TeX: {paths['tex']}")

BUNDLE_DIR = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_bundle"

bundle_paths = save_full_evaluation_bundle(
    MAP_RESULTS,
    BUNDLE_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=EVAL_SEEDS,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(f"Saved bundle dir: {bundle_paths['dir']}")
print(f"Saved seed summary: {bundle_paths['seed_summary']}")
print(f"Saved model summary: {bundle_paths['model_summary']}")


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,0.661458,0.407849,1.580521,5.184420e-01,36.953125,4.448627,3,3,64
1,6h_vs_8z,vanilla,vanilla,pci-ippo,0.984375,0.015625,1.980599,1.985854e-02,47.479167,11.287971,3,3,64
2,6h_vs_8z,vanilla,vanilla,mappo,0.932292,0.045105,1.926536,4.976182e-02,37.916667,4.340161,3,3,64
3,6h_vs_8z,vanilla,vanilla,pci-mappo,1.000000,0.000000,2.000000,1.493992e-08,32.677083,1.010846,3,3,64
4,6h_vs_8z,tar_w=6,tar_w=6,ippo,0.453125,0.291899,1.358854,3.491679e-01,42.838542,5.867106,3,3,64
5,6h_vs_8z,tar_w=6,tar_w=6,pci-ippo,0.869792,0.212140,1.849635,2.467656e-01,46.494792,10.365505,3,3,64
6,6h_vs_8z,tar_w=6,tar_w=6,mappo,0.838542,0.252591,1.817943,2.848890e-01,38.182292,1.067124,3,3,64
7,6h_vs_8z,tar_w=6,tar_w=6,pci-mappo,0.807292,0.168528,1.790911,1.835380e-01,43.348958,6.283238,3,3,64


\begin{table*}[t]
\centering
\caption{Deterministic SMAX win rate using final checkpoints. Rows are grouped by map and training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:smax_cross_map_final}
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Map & Constraint & ippo & pci-ippo & mappo & pci-mappo \\
\midrule
6h_vs_8z & vanilla & 66.1 \scriptsize$\pm$ 40.8 & 98.4 \scriptsize$\pm$ 1.6 & 93.2 \scriptsize$\pm$ 4.5 & 100.0 \scriptsize$\pm$ 0.0 \\
6h_vs_8z & tar_w=6 & 45.3 \scriptsize$\pm$ 29.2 & 87.0 \scriptsize$\pm$ 21.2 & 83.9 \scriptsize$\pm$ 25.3 & 80.7 \scriptsize$\pm$ 16.9 \\
\bottomrule
\end{tabular}%
}
\end{table*}
Saved CSV: /home/chamjin/research/pci_marl/figures/smax_checkpoint_eval_final_summary.csv
Saved TeX: /home/chamjin/research/pci_marl/figures/smax_checkpoint_eval_final_table.tex
Saved bundle dir: /home/chamjin/research/pci_marl/figures/smax_checkpoint_eval_final_bundle
Saved seed summary: /home/chamjin/research/pci_marl/figur

In [16]:
if not MAP_RESULTS:
    raise RuntimeError("Run the full evaluation cell first.")

combined_seed_summary = pd.concat(
    [MAP_RESULTS[map_name]["seed_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_seed_summary)

seedwise_group_a = ["3m", "2s3z", "3s5z", "6h_vs_8z",  "smacv2_5_units", "smacv2_10_units"]

seedwise_latex_a = scu.format_seedwise_cross_map_latex(
    combined_seed_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=seedwise_group_a,
    model_order=TARGET_MODELS,
    caption=(
        "Seed-wise deterministic SMAX test win rates (\\%) using final checkpoints for "
        "\\texttt{3m}, \\texttt{2s3z}, and \\texttt{3s5z}. Each entry is the mean over 64 evaluation seeds. "
        "Missing entries indicate that the corresponding training seed was unavailable."
    ),
    label="tab:smax_seedwise_part1",
)
print(seedwise_latex_a)

seedwise_latex_b = scu.format_seedwise_cross_map_latex(
    combined_seed_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=seedwise_group_b,
    model_order=TARGET_MODELS,
    caption=(
        "Seed-wise deterministic SMAX test win rates (\\%) using final checkpoints for "
        "\\texttt{smacv2\\_5\\_units}, \\texttt{smacv2\\_10\\_units}, and \\texttt{6h\\_vs\\_8z}. "
        "Each entry is the mean over 64 evaluation seeds. Missing entries indicate that the corresponding training seed was unavailable."
    ),
    label="tab:smax_seedwise_part2",
)
print(seedwise_latex_b)

seedwise_latex_a_path = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_seedwise_part1.tex"
seedwise_latex_b_path = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_seedwise_part2.tex"
seedwise_latex_a_path.write_text(seedwise_latex_a)
seedwise_latex_b_path.write_text(seedwise_latex_b)
print(f"Saved TeX A: {seedwise_latex_a_path}")
print(f"Saved TeX B: {seedwise_latex_b_path}")


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,6h_vs_8z,vanilla,vanilla,ippo,1,0.796875,1.761797,41.875000,1,64
1,6h_vs_8z,vanilla,vanilla,ippo,2,0.984375,1.983984,33.218750,1,64
2,6h_vs_8z,vanilla,vanilla,ippo,42,0.203125,0.995781,35.765625,1,64
3,6h_vs_8z,vanilla,vanilla,pci-ippo,1,1.000000,2.000000,34.531250,1,64
4,6h_vs_8z,vanilla,vanilla,pci-ippo,2,0.968750,1.960312,52.656250,1,64
5,6h_vs_8z,vanilla,vanilla,pci-ippo,42,0.984375,1.981484,55.250000,1,64
6,6h_vs_8z,vanilla,vanilla,mappo,1,0.906250,1.896797,39.093750,1,64
7,6h_vs_8z,vanilla,vanilla,mappo,2,0.984375,1.983984,33.109375,1,64
8,6h_vs_8z,vanilla,vanilla,mappo,42,0.906250,1.898828,41.546875,1,64
9,6h_vs_8z,vanilla,vanilla,pci-mappo,1,1.000000,2.000000,31.875000,1,64


\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Seed-wise deterministic SMAX test win rates (\%) using final checkpoints for \texttt{3m}, \texttt{2s3z}, and \texttt{3s5z}. Each entry is the mean over 64 evaluation seeds. Missing entries indicate that the corresponding training seed was unavailable.}
\label{tab:smax_seedwise_part1}
\begin{tabular}{llccccc}
\toprule
Map & Constraint & Seed & IPPO & PCI-IPPO & MAPPO & PCI-MAPPO \\
\midrule
\midrule
\midrule
\midrule
6h_vs_8z & vanilla & 1 & 79.7 & 100.0 & 90.6 & 100.0 \\
 &  & 2 & 98.4 & 96.9 & 98.4 & 100.0 \\
 &  & 42 & 20.3 & 98.4 & 90.6 & 100.0 \\
 & tar_w=6 & 1 & 18.8 & 100.0 & 54.7 & 73.4 \\
 &  & 2 & 40.6 & 98.4 & 98.4 & 100.0 \\
 &  & 42 & 76.6 & 62.5 & 98.4 & 68.8 \\
\midrule
\midrule
\bottomrule
\end{tabular}
\end{table*}


NameError: name 'seedwise_group_b' is not defined